In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# Настройки отображения для графиков и таблиц
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)



In [ ]:
!pip install gdown -q

In [ ]:
FILE_ID = '1TLBoYoalbs7V5cCxeRkRMqPI9P0v2Fvy'
!gdown --id {FILE_ID} -O meta.csv

In [ ]:

df = pd.read_csv('/content/meta.csv')
display(df.head())

In [ ]:
#размер таблицы
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} колонок")

#типы данных и пропуски (NaN)
df.info()

print("\n     Пропущенные значения")
print(df.isnull().sum())

In [ ]:
#bs файл
FILE_ID_BS = '1A5O-fLKcc6FwfeVU01kpps2biMx6DAmJ'
!gdown --id {FILE_ID_BS} -O meta_bs.csv

In [ ]:
#загружаем
df_bs = pd.read_csv('meta_bs.csv')

print(f"Размер BS датасета: {df_bs.shape[0]} строк, {df_bs.shape[1]} колонок")
display(df_bs.head())

print("\n    Пропуски в BS")
print(df_bs.isnull().sum())

#какие типы чипов
if 'kind' in df_bs.columns:
    print("\nТипы чипов:")
    print(df_bs['kind'].value_counts())

In [ ]:
# Настройки графиков
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

#данные
df_bs = pd.read_csv('meta_bs.csv')
df_bs_train = df_bs[df_bs['burn_area_ha'].notna()].copy()

#распределение площади гари
plt.figure(figsize=(10, 4))
sns.histplot(df_bs_train['burn_area_ha'], bins=30, kde=True, color='crimson')
plt.title('Распределение площади гари в BS-чипах (Train)')
plt.xlabel('Площадь гари (гектары)')
plt.ylabel('Частота')
plt.show()

print("    Статистика площади гари (га)")
print(df_bs_train['burn_area_ha'].describe())

#распределение классов степени поражения
sev_cols = ['sev1_px', 'sev2_px', 'sev3_px']
sev_sums = df_bs_train[sev_cols].sum()
sev_perc = sev_sums / sev_sums.sum() * 100

print("\n     Распределение классов степени поражения (Train)")
print(sev_perc.round(1))

plt.figure(figsize=(8, 5))
sev_perc.plot(kind='bar', color=['#8bc34a', '#ff9800', '#f44336'])
plt.title('Доля пикселей по степеням поражения (Train)')
plt.ylabel('Процент от всех пикселей гари')
plt.xticks(rotation=0)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

df_af = pd.read_csv('meta.csv')
df_bs = pd.read_csv('meta_bs.csv')

print("=" * 60)
print("ПРОВЕРКА AF-ДАННЫХ")
print("=" * 60)

#уникальность chip_id
print(f"Уникальных chip_id: {df_af['chip_id'].nunique()} из {len(df_af)}")
if df_af['chip_id'].nunique() != len(df_af):
    print("есть дупликат chip_id")
else:
    print("дубликатов нет")

# 2. kind
print(f"\nТипы kind: {df_af['kind'].unique()}")

# 3. Размеры и gsd
print(f"\nwidth:  {df_af['width'].unique()}")
print(f"height: {df_af['height'].unique()}")
print(f"gsd:    {df_af['gsd'].unique()}")

# 5. valid_frac в [0,1]?
bad_valid = df_af[(df_af['valid_frac'] < 0) | (df_af['valid_frac'] > 1)]
print(f"Чипов с valid_frac вне [0,1]: {len(bad_valid)}")

# 6. n_fire_px: неотрицательные, целые?
print(f"\nn_fire_px — min: {df_af['n_fire_px'].min()}, max: {df_af['n_fire_px'].max()}")
print(f"Дробных n_fire_px: {(df_af['n_fire_px'] % 1 != 0).sum()}")
print(f"Отрицательных n_fire_px: {(df_af['n_fire_px'] < 0).sum()}")

# 7. Спутники
print(f"\nСпутники: {df_af['satellite'].value_counts().to_dict()}")

# 8. Даты
df_af['acq_datetime'] = pd.to_datetime(df_af['acq_datetime'])
print(f"\nДиапазон дат: {df_af['acq_datetime'].min()} — {df_af['acq_datetime'].max()}")

# 9. EPSG
print(f"\nEPSG: {df_af['epsg'].unique()}")


print("\n" + "=" * 60)
print("ПРОВЕРКА BS-ДАННЫХ")
print("=" * 60)

# 1. Уникальность chip_id
print(f"Уникальных chip_id: {df_bs['chip_id'].nunique()} из {len(df_bs)}")

# 2. kind
print(f"Типы kind: {df_bs['kind'].unique()}")

# 3. Размеры
print(f"\nwidth:  {df_bs['width'].unique()}")
print(f"height: {df_bs['height'].unique()}")
print(f"gsd:    {df_bs['gsd'].unique()}")

# 4. Координаты
bad_coords_bs = df_bs[(df_bs['x_min'] >= df_bs['x_max']) | (df_bs['y_min'] >= df_bs['y_max'])]
print(f"\nЧипов с неправильными координатами: {len(bad_coords_bs)}")

# 5. valid_frac и cloud_frac
print(f"\nvalid_frac — min: {df_bs['valid_frac'].min()}, max: {df_bs['valid_frac'].max()}")
print(f"cloud_frac — min: {df_bs['cloud_frac'].min()}, max: {df_bs['cloud_frac'].max()}")
print(f"valid_frac + cloud_frac > 1.01: {((df_bs['valid_frac'] + df_bs['cloud_frac']) > 1.01).sum()}")

# 6. Даты: pre < post?
df_bs['date_pre'] = pd.to_datetime(df_bs['date_pre'])
df_bs['date_post'] = pd.to_datetime(df_bs['date_post'])
df_bs['s1_date_pre'] = pd.to_datetime(df_bs['s1_date_pre'])
df_bs['s1_date_post'] = pd.to_datetime(df_bs['s1_date_post'])

bad_dates = df_bs[df_bs['date_pre'] >= df_bs['date_post']]
print(f"\nЧипов, где date_pre >= date_post: {len(bad_dates)}")

bad_s1_dates = df_bs[df_bs['s1_date_pre'] >= df_bs['s1_date_post']]
print(f"Чипов, где s1_date_pre >= s1_date_post: {len(bad_s1_dates)}")

# 7. Проверка арифметики площади
# Площадь одного пикселя Sentinel-2 при gsd=20м = 20*20 = 400 м² = 0.04 га
total_px = df_bs['sev1_px'] + df_bs['sev2_px'] + df_bs['sev3_px']
expected_area = total_px * 0.04  # в гектарах
diff = (df_bs['burn_area_ha'] - expected_area).abs()
print(f"\nМаксимальное расхождение burn_area_ha и расчётной площади: {diff.max():.4f} га")
print(f"Чипов с расхождением > 1 га: {(diff > 1).sum()}")

# 8. fire_event_id
print(f"\nУникальных fire_event_id: {df_bs['fire_event_id'].nunique()}")
print(f"Топ-5 пожаров по числу чипов:")
print(df_bs['fire_event_id'].value_counts().head())

# 9. EPSG
print(f"\nEPSG: {df_bs['epsg'].unique()}")

Каждый чип BS относится к уникальному fire_event_id (224 события на 224 чипа). Это гарантирует отсутствие утечки между train и val, но ограничивает проверку обобщения внутри одного пожара. Используем StratifiedKFold по распределению классов степени поражения


Самое важное наблюдение по модулю активного горения - экстремальный дисбаланс. В самом «пожарном» чипе из всего набора огонь занимает 243 пикселя из 65 536. Это 0.37%. А медиана вообще смешная - 21 пиксель, то есть 0.03%.

т.е на 3000 пикселей фона приходится 1 пиксель огня.

Если обучать модель «в лоб» на обычной CrossEntropy, она быстро поймёт, что выгоднее всего предсказывать везде ноль. Формально точность будет 99.97%, но пожары она не найдёт никогда. А метрика F1 в таком случае обнулится, и мы потеряем 35% итогового Score.

Что будем делать: использовать Focal Loss (она автоматически усиливает вклад редких классов) плюс WeightedRandomSampler, чтобы чипы с огнём попадали в батч не реже, чем чипы без огня.

съёмка идёт с трёх спутников (SNPP, NOAA-20, NOAA-21). У них разная калибровка каналов, поэтому нормализовать яркостную температуру нужно с учётом сенсора. Либо подавать идентификатор спутника как отдельный вход.

Отдельное ограничение — в AF-метаданных нет fire_event_id. Это значит, что мы не можем использовать GroupKFold и не можем гарантировать, что чипы одного и того же пожара не попадут одновременно в train и val. В отчёте это фиксируем как известное ограничение и используем StratifiedKFold по наличию огня.

In [ ]:
!pip install rasterio torch -q

In [ ]:
%%writefile dataset.py
"""
dataset.py — Модуль загрузки данных.
Версия БЕЗ подпапок: все файлы (pre, post, mask) лежат в одной папке.
"""

import os
import numpy as np
import pandas as pd
import rasterio
import torch
from torch.utils.data import Dataset

EXT = '.tif'


def _safe_dnbr(b8a_pre, b12_pre, b8a_post, b12_post, eps=1e-8):
    nbr_pre = (b8a_pre - b12_pre) / (b8a_pre + b12_pre + eps)
    nbr_post = (b8a_post - b12_post) / (b8a_post + b12_post + eps)
    dnbr = nbr_pre - nbr_post
    dnbr = np.nan_to_num(dnbr, nan=0.0, posinf=0.0, neginf=0.0)
    dnbr = np.clip(dnbr, -1.0, 1.0)
    return dnbr.astype(np.float32)


def _load_tif(path):
    with rasterio.open(path) as src:
        return src.read()


class FireDatasetBS(Dataset):
    """
    Все файлы лежат в ОДНОЙ папке (data_dir):
      {chip_id}_Sentinel-2_pre{EXT}
      {chip_id}_Sentinel-2_post{EXT}
      {chip_id}_MASK{EXT}
    """
    def __init__(self, meta_df, data_dir, transform=None, return_mask=True):
        self.meta = meta_df.reset_index(drop=True)
        self.data_dir = data_dir
        self.transform = transform
        self.return_mask = return_mask

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        chip_id = self.meta.iloc[idx]['chip_id']

        pre_path  = os.path.join(self.data_dir, f'{chip_id}_Sentinel-2_pre{EXT}')
        post_path = os.path.join(self.data_dir, f'{chip_id}_Sentinel-2_post{EXT}')
        mask_path = os.path.join(self.data_dir, f'{chip_id}_MASK{EXT}')

        pre  = _load_tif(pre_path).astype(np.float32)
        post = _load_tif(post_path).astype(np.float32)

        pre  = pre[:9]  / 10000.0
        post = post[:9] / 10000.0

        dnbr = _safe_dnbr(pre[6], pre[8], post[6], post[8])
        image = np.concatenate([pre, post, dnbr[np.newaxis, :, :]], axis=0)

        if not self.return_mask:
            if self.transform:
                image = self.transform(image)
            return torch.from_numpy(image).float()

        mask = _load_tif(mask_path)[0].astype(np.int64)

        if self.transform:
            image, mask = self.transform(image, mask)

        return torch.from_numpy(image).float(), torch.from_numpy(mask).long()

In [ ]:
!ls -la "/content/drive/MyDrive/sample/"

In [ ]:
!ls -la /content/dataset.py

In [ ]:
import sys
sys.path.append('/content')
from dataset import FireDatasetBS
import pandas as pd
from torch.utils.data import DataLoader

# Создаём «фейковые» метаданные для одного чипа
df = pd.DataFrame({'chip_id': ['BS_tr_000001']})

DATA_DIR = '/content/drive/MyDrive/sample'

ds = FireDatasetBS(df, DATA_DIR)
loader = DataLoader(ds, batch_size=1)

for img, mask in loader:
    print(f"✅ image shape: {img.shape}, dtype: {img.dtype}")
    print(f"✅ mask shape:  {mask.shape}, dtype: {mask.dtype}")
    print(f"✅ Уникальные классы в маске: {mask.unique()}")
    break

In [ ]:
import matplotlib.pyplot as plt
from dataset import FireDatasetBS
import pandas as pd

df = pd.DataFrame({'chip_id': ['BS_tr_000001']})
ds = FireDatasetBS(df, '/content/drive/MyDrive/sample')
img, mask = ds[0]  # img: (19, 512, 512), mask: (512, 512)

# Распаковываем: первые 9 каналов — pre, следующие 9 — post, последний — dNBR
pre = img[:9].numpy()
post = img[9:18].numpy()
dnbr = img[18].numpy()
mask = mask.numpy()

fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# RGB "До" (B4=2, B3=1, B2=0)
rgb_pre = np.stack([pre[2], pre[1], pre[0]], axis=-1)
axes[0].imshow(np.clip(rgb_pre, 0, 1))
axes[0].set_title('"До" пожара')

# RGB "После"
rgb_post = np.stack([post[2], post[1], post[0]], axis=-1)
axes[1].imshow(np.clip(rgb_post, 0, 1))
axes[1].set_title('"После" пожара')

# dNBR
im3 = axes[2].imshow(dnbr, cmap='RdBu_r', vmin=-0.5, vmax=1.0)
axes[2].set_title('dNBR (красное = гарь)')
plt.colorbar(im3, ax=axes[2], fraction=0.046)

# Маска
im4 = axes[3].imshow(mask, cmap='viridis', vmin=0, vmax=3)
axes[3].set_title('Эталонная маска (0-3)')
plt.colorbar(im4, ax=axes[3], fraction=0.046)

plt.tight_layout()
plt.show()

На одном чипе BS (BS_tr_000001) проверена загрузка через класс FireDatasetBS. Возвращаемые тензоры имеют форму (19, 512, 512) и (512, 512) с типами float32 и int64 соответственно. Каналы собираются как [pre_B2..B12 (9), post_B2..B12 (9), dNBR (1)]. Нормализация Sentinel-2 L2A выполнена делением на 10000. Значения dNBR ограничены диапазоном [-1, 1]. Эталонная маска содержит классы 0–3.